In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
import joblib
import os


In [9]:
# --- Step 1: Load the dataset ---
# Make sure your 'crop_data.csv' is in the 'data/' directory at the project root
# If you place this script in 'backend/notebooks/', the path will be '../../data/crop_data.csv'
# If you place this script at the root, it's 'data/crop_data.csv'
# Assuming you run this from the project root (e.g., in notebooks/model_training.ipynb)
# If running as a script from 'backend/' then adjust the path to '../data/crop_data.csv'
df = pd.read_csv('../data/crop_data.csv')

print("Dataset loaded successfully. First 5 rows:")
print(df.head())



Dataset loaded successfully. First 5 rows:
  crop_name   region  season soil_type weather_condition  rainfall_mm  \
0    Cotton     East  Summer     Sandy             Humid   377.015503   
1    Barley     West  Spring     Peaty             Humid   323.071224   
2   Soybean  Central  Spring     Sandy             Humid   275.346521   
3    Cotton     East  Spring     Sandy             Rainy    90.884311   
4     Maize    North  Winter     Sandy               Dry   180.806731   

   temperature_c  humidity_percent  fertilizer_used_kg  market_demand  \
0      21.195504         31.926707          139.369397    7075.306673   
1      39.620806         31.970201          206.762354    4158.673939   
2      35.606975         73.295710          272.085804    4072.058985   
3      21.638990         41.587635          209.483749    9916.756596   
4      37.417497         69.131908          191.464314    7469.332505   

   supply_quantity   crop_price  
0      1895.984590  3984.483236  
1      2484

In [10]:
# --- Step 2: Prepare Features and Target ---
X = df.drop('crop_price', axis=1)
y = df['crop_price']



In [11]:
# Identify categorical and numerical columns based on your dataset
categorical_features = ['crop_name', 'region', 'season', 'soil_type', 'weather_condition']
numerical_features = ['rainfall_mm', 'temperature_c', 'humidity_percent', 'fertilizer_used_kg', 'market_demand', 'supply_quantity']


In [12]:
# --- Step 3: Create a Preprocessing and Model Pipeline ---
# Use ColumnTransformer to apply OneHotEncoder to categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep numerical features as they are
)


In [13]:
# Create a Pipeline that first preprocesses and then trains the model
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

print("\nPipeline created: Preprocessor (OneHotEncoder) -> RandomForestRegressor")



Pipeline created: Preprocessor (OneHotEncoder) -> RandomForestRegressor


In [14]:
# --- Step 4: Split Data and Train Model ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining the RandomForestRegressor model...")
model_pipeline.fit(X_train, y_train)
print("Model training complete.")


Training the RandomForestRegressor model...
Model training complete.


In [15]:
# --- Step 5: Evaluate the Model (Optional, but Recommended) ---
y_pred = model_pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"\nModel Evaluation:")
print(f"R-squared (R2): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")




Model Evaluation:
R-squared (R2): 0.8322
Mean Absolute Error (MAE): 270.20


In [16]:
# --- Step 6: Save the Trained Model Pipeline to a .pkl file ---
model_dir = 'models' # This refers to the 'models/' directory at the project root
os.makedirs(model_dir, exist_ok=True) # Create the directory if it doesn't exist
model_path = os.path.join(model_dir, 'crop_prediction_model_pipeline.pkl')
joblib.dump(model_pipeline, model_path)

print(f"\nModel pipeline saved to: {model_path}")




Model pipeline saved to: models\crop_prediction_model_pipeline.pkl


In [17]:
# Display the feature names after one-hot encoding for debugging/understanding (optional)
# This shows the order and names of features the model expects after preprocessing
ohe_feature_names = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = list(ohe_feature_names) + numerical_features
print("\nExpected feature names for prediction after preprocessing:")
print(all_feature_names)


Expected feature names for prediction after preprocessing:
['crop_name_Barley', 'crop_name_Cotton', 'crop_name_Maize', 'crop_name_Millet', 'crop_name_Rice', 'crop_name_Soybean', 'crop_name_Sugarcane', 'crop_name_Wheat', 'region_Central', 'region_East', 'region_North', 'region_South', 'region_West', 'season_Autumn', 'season_Monsoon', 'season_Spring', 'season_Summer', 'season_Winter', 'soil_type_Clayey', 'soil_type_Loamy', 'soil_type_Peaty', 'soil_type_Sandy', 'soil_type_Silty', 'weather_condition_Cold', 'weather_condition_Dry', 'weather_condition_Humid', 'weather_condition_Moderate', 'weather_condition_Rainy', 'rainfall_mm', 'temperature_c', 'humidity_percent', 'fertilizer_used_kg', 'market_demand', 'supply_quantity']
